In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

C:\Users\Admin\AppData\Local\Temp\ipykernel_24376\2191127060.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
E:\Gen_AI\GEN_AI_Project\RAG_Mini_Project\Mini_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_PATH = "Statistics_syllabus.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages.")

Loaded 17 pages.


In [4]:
print(pages[9].page_content)

S. Y. B.Sc. (Statistics) 
 
MES ABASAHEB GARWARE COLLEGE, PUNE 4 (AUTONOMOUS)                                                               10  
 
 Paper Title - Practical Paper - I 
 
Course Outcomes: 
At the end of this course, students are able to… 
1)  Learn to fit a suitable discrete and continuous probability distributions to the data.  
2)  Identify the suitable probability model for the population.  
3)  Generate random samples from the continuous probability distributions.  
4)  Learn the basic commands of R-software. Carry out data visualization, summary statistics, 
computation of probabilities for discrete distributions using R software.  
 
Expt. No. Title of experiment 
1 Fitting of negative binomial distribution and compu tation of expected frequencies. Plot 
of expected frequencies vs. observed frequencies. 
2 Fitting of normal distribution and computation of expected frequencies. Plot of expected 
frequencies vs. observed frequencies. 
3 Applications of negative binomi

In [10]:
#create structure based chunks
experiment_chunks = []

text = pages[9].page_content
lines = text.split("\n")

current_experiment = None
current_text = []

for line in lines:

    line = line.strip()

    if not line:
        continue

    # Case 1: Experiment number and title are on the same line
    match = re.match(r"^(\d{1,2})\s+(.*)", line)

    # Case 2: Experiment number appears alone on a line
    standalone = re.match(r"^(\d{1,2})$", line)

    if match or standalone:

        # Save previous experiment
        if current_experiment is not None:

            experiment_chunks.append(
                Document(
                    page_content=(
                        f"Practical Paper-I. "
                        f"Experiment {current_experiment}: "
                        + " ".join(current_text)
                    ),
                    metadata={
                        "page": pages[9].metadata.get("page", 0) + 1,
                        "type": "experiment",
                        "experiment_number": current_experiment
                    }
                )
            )

        # Start new experiment
        if match:
            current_experiment = match.group(1)
            current_text = [match.group(2)]

        else:
            current_experiment = standalone.group(1)
            current_text = []

    else:

        # Continue adding text to current experiment
        if current_experiment is not None:
            current_text.append(line)


# Save the last experiment
if current_experiment is not None:

    experiment_chunks.append(
        Document(
            page_content=(
                f"Practical Paper-I. "
                f"Experiment {current_experiment}: "
                + " ".join(current_text)
            ),
            metadata={
                "page": pages[9].metadata.get("page", 0) + 1,
                "type": "experiment",
                "experiment_number": current_experiment
            }
        )
    )


print("Number of experiment chunks:", len(experiment_chunks))

Number of experiment chunks: 11


In [12]:
for chunk in experiment_chunks:
    print(chunk.metadata["experiment_number"], type(chunk.metadata["experiment_number"]))

1 <class 'str'>
2 <class 'str'>
3 <class 'str'>
4 <class 'str'>
5 <class 'str'>
6 <class 'str'>
7 <class 'str'>
8 <class 'str'>
9 <class 'str'>
10 <class 'str'>
11 <class 'str'>


In [15]:
print("Number of experiment chunks:", len(experiment_chunks))

for chunk in experiment_chunks:
    print("\n" + "=" * 80)
    print("Experiment:", chunk.metadata["experiment_number"])
    print(chunk.page_content)

Number of experiment chunks: 11

Experiment: 1
Practical Paper-I. Experiment 1: Fitting of negative binomial distribution and compu tation of expected frequencies. Plot of expected frequencies vs. observed frequencies.

Experiment: 2
Practical Paper-I. Experiment 2: Fitting of normal distribution and computation of expected frequencies. Plot of expected frequencies vs. observed frequencies.

Experiment: 3
Practical Paper-I. Experiment 3: Applications of negative binomial and multinomial  distributions.

Experiment: 4
Practical Paper-I. Experiment 4: Applications of exponential and normal distributi ons.

Experiment: 5
Practical Paper-I. Experiment 5: A) Generating random samples from exponential distribution using distribution function method. B) Generating random samples from normal and log normal distributions using i) distribution function  ii) Box-Muller transformation.

Experiment: 6
Practical Paper-I. Experiment 6: Applications of truncated binomial, truncated Poi sson and trunca

In [11]:
#check experiment 9
for chunk in experiment_chunks:

    if chunk.metadata["experiment_number"] == "9":

        print("FOUND EXPERIMENT 9")
        print("\nPage:", chunk.metadata["page"])
        print("\nChunk:")
        print(chunk.page_content)

FOUND EXPERIMENT 9

Page: 10

Chunk:
Practical Paper-I. Experiment 9: Finding summary statistics using summary( ) and fivenum( ) functions. Calculate arithmetic mean (A.M.), geometric mean (G .M.), harmonic mean (H.M.), median, mode, quantiles, range, quartile deviation (Q.D.), variance, coefficient of variation (C.V.) (ungrouped data) using R software.


In [17]:
#To see what type of pages a
for i, page in enumerate(pages):
    print("\n" + "=" * 80)
    print("PAGE:", i + 1)
    print("=" * 80)

    # Print first 1000 characters of each page
    print(page.page_content[:1000])


PAGE: 1
Maharashtra Education Society’s 
ABASAHEB GARWARE COLLEGE (AUTONOMOUS) 
KARVE ROAD, PUNE 411004 
(Affiliated to Savitribai Phule Pune University) 
 
 
Three year B. Sc. Degree Program in Statistics  
(Faculty of Science and Technology) 
 
 
Syllabus under Autonomy  
S. Y. B. Sc. (Statistics) 
 
Choice Based Credit System (C. B. C. S.) Syllabus 
To be implemented from Academic Year 2023-2024

PAGE: 2
TITLE OF THE COURSE : S.  Y.  B.  SC. (S TATISTICS ) 
 
ELIGIBILITY:  
1.  Passed (with at least 22 credits) in F. Y. B. Sc. with Statistics as one of the subjects.  
2.  A student of the three year B. Sc. degree course wi ll not be allowed to offer Statistics 
and Statistical Techniques simultaneously in any of the three years of the course. 
 
STRUCTURE OF THE COURSE: (Each theory and practical paper has 2 credit)  
 
 
Semester  
 
Course Type  
 
Course 
Code  
 
Course Title  
 
Remark  
No. of 
Lectures  
/Practicals to be 
conducted  
      III  
Core Course 
USST -231  Prob